# ViT from scratch: tasks

You build a Vision Transformer piece by piece, train it on CIFAR-10, look inside it, and finally load real CLIP ViT-B/32 weights into your own code.

**How each task works**

1. A markdown cell says what to build and the exact input/output shapes.
2. A code cell has a `TODO` and `raise NotImplementedError`. Replace it with your code.
3. The next cell is a **check**. It compares your code against PyTorch's own layers. Run it: `OK` means you're done.

Stuck? `from vit_solutions import patchify` (or any other name) swaps in the reference version so you can keep going. Try for real first.

**Where to run**: on the devbox. Tasks 1 to 7 run on CPU in seconds. Tasks 8 to 12 want a GPU worker.

```bash
cd /mlx_devbox/users/liyangsun/playground/repos/cvpr_qoe/tutorials
pip install jupyterlab matplotlib   # once
jupyter lab --no-browser --port 8888
```

| Task | What you build | Ties to |
|---|---|---|
| 1 | `patchify` with reshape/permute | the shape bookkeeping everything else uses |
| 2 | `PatchEmbed` with `Conv2d` | Swin `PatchEmbed`, `vit_encoder.py` step 1 |
| 3 | `[CLS]` token + position embedding | `vit_encoder.py` steps 2-3 |
| 4 | scaled dot-product attention | `F.scaled_dot_product_attention` |
| 5 | multi-head self-attention | `nn.MultiheadAttention` |
| 6 | MLP + pre-norm transformer block | `nn.TransformerEncoderLayer`, `train_theta.py` fusion |
| 7 | the full `ViT` | |
| 8 | CIFAR-10 data pipeline | |
| 9 | training loop, weight-decay groups, LR schedule | `train_theta.py` |
| 10 | attention maps, position-embedding similarity | |
| 11 | load CLIP ViT-B/32 weights into your ViT | `vit_encoder.py` |
| 12 | stretch: fine-tune pretrained vs from scratch | the Stage 1 encoder decision |

In [ ]:
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(torch.__version__, device)


def check(name, cond, msg=''):
    if not cond:
        raise AssertionError(f'{name}: FAILED {msg}')
    print(f'{name}: OK')

## Task 1: patchify

A ViT treats an image as a sequence of patches. Cut a `(B, C, H, W)` image into non-overlapping `p x p` patches and flatten each one.

- Output: `(B, N, C*p*p)` with `N = (H/p) * (W/p)`.
- Patch order: row-major over the grid (left to right, then top to bottom).
- Inside a patch: flatten in `(C, p, p)` order. That's the order `Conv2d` weights use, which matters in Task 2.
- Only `reshape` and `permute`. No loops, no `unfold`.

Hint: split `H` into `(H/p, p)` and `W` into `(W/p, p)`, move the two grid axes next to the batch axis, then flatten.

In [ ]:
def patchify(x: torch.Tensor, p: int) -> torch.Tensor:
    """(B, C, H, W) -> (B, N, C*p*p)."""
    # TODO
    raise NotImplementedError

In [ ]:
x = torch.randn(2, 3, 32, 32)
out = patchify(x, 4)
check('shape', out.shape == (2, 64, 48), out.shape)
ref = F.unfold(x, kernel_size=4, stride=4).transpose(1, 2)
check('matches F.unfold', torch.allclose(out, ref))
check('patch 0 is the top-left corner', torch.equal(out[0, 0], x[0, :, :4, :4].reshape(-1)))
check('patch 1 is right of it', torch.equal(out[0, 1], x[0, :, :4, 4:8].reshape(-1)))

**Q1.** How many patch tokens does a 224x224 image give with patch 16? With patch 32? Our Stage 1 encoder is ViT-B/32: how many tokens go through its transformer per frame, counting `[CLS]`?

*Your answer:*

## Task 2: PatchEmbed with Conv2d

A linear layer on each flattened patch is the same thing as a `Conv2d` whose kernel size and stride both equal the patch size. Real ViTs use the conv version.

- `self.proj`: a `Conv2d` from `in_chans` to `dim`, kernel = stride = `patch_size`, with `bias=bias`.
- `self.num_patches`: `N`.
- `forward`: `(B, C, H, W) -> (B, N, dim)`. The conv gives `(B, dim, H/p, W/p)`; turn that into a token sequence.

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_chans=3, dim=192, bias=True):
        super().__init__()
        # TODO: self.num_patches, self.proj
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
pe = PatchEmbed(32, 4, 3, 192)
x = torch.randn(2, 3, 32, 32)
out = pe(x)
check('shape', out.shape == (2, 64, 192), out.shape)
check('num_patches', pe.num_patches == 64)

# Same weights as a Linear on patchify(x): conv weight (D, C, p, p) -> (D, C*p*p)
lin = nn.Linear(48, 192)
with torch.no_grad():
    lin.weight.copy_(pe.proj.weight.reshape(192, -1))
    lin.bias.copy_(pe.proj.bias)
check('conv == linear on patches', torch.allclose(out, lin(patchify(x, 4)), atol=1e-5))

**Q2.** Why does the conv equal a linear layer on patches here? What changes if `stride < kernel_size`? (Hint: some early CNN-ViT hybrids do this on purpose.)

*Your answer:*

## Task 3: [CLS] token and position embedding

Attention has no notion of order: shuffle the tokens and the outputs shuffle with them. Position embeddings fix that. A learned `[CLS]` token is put in front; after the last block its output summarises the image.

- `self.cls`: `nn.Parameter` of shape `(1, 1, dim)`.
- `self.pos`: `nn.Parameter` of shape `(1, num_patches + 1, dim)`.
- Initialise both with `nn.init.trunc_normal_(..., std=0.02)`.
- `forward`: `(B, N, dim) -> (B, N + 1, dim)`: put `[CLS]` first, then add `pos` to every token.

Hint: `expand` copies the `[CLS]` token across the batch without allocating memory.

In [ ]:
class TokenPrep(nn.Module):
    def __init__(self, num_patches: int, dim: int):
        super().__init__()
        # TODO: self.cls, self.pos
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
tp = TokenPrep(64, 192)
x = torch.randn(3, 64, 192)
out = tp(x)
check('shape', out.shape == (3, 65, 192), out.shape)
check('cls first, same for every image', torch.allclose(out[:, 0], (tp.cls + tp.pos[:, 0]).expand(3, -1)))
check('patches keep order', torch.allclose(out[:, 1:], x + tp.pos[:, 1:]))
check('cls and pos are learnable', {n for n, _ in tp.named_parameters()} == {'cls', 'pos'})

## Task 4: scaled dot-product attention

$\text{attn} = \text{softmax}(q k^T / \sqrt{d})$, output $= \text{attn} \cdot v$.

- `q, k, v`: `(B, h, T, d)`.
- `mask`: optional bool, broadcastable to `(B, h, T, T)`, `True` = this key may be attended to. Masked scores become `-inf` before the softmax.
- Return `(out, attn)`: `out` is `(B, h, T, d)`, `attn` is `(B, h, T, T)`. Keep `attn`; Task 10 visualises it.

In [ ]:
def attention(q, k, v, mask=None):
    # TODO
    raise NotImplementedError

In [ ]:
q, k, v = (torch.randn(2, 3, 10, 16) for _ in range(3))
out, attn = attention(q, k, v)
check('shapes', out.shape == (2, 3, 10, 16) and attn.shape == (2, 3, 10, 10))
check('rows sum to 1', torch.allclose(attn.sum(-1), torch.ones(2, 3, 10), atol=1e-5))
check('matches F.scaled_dot_product_attention', torch.allclose(out, F.scaled_dot_product_attention(q, k, v), atol=1e-5))

# Key padding: the last 4 keys of image 1 are padding.
mask = torch.ones(2, 1, 1, 10, dtype=torch.bool)
mask[1, ..., 6:] = False
out_m, attn_m = attention(q, k, v, mask)
check('masked keys get zero weight', attn_m[1, ..., 6:].abs().max() == 0)
check('masked matches torch', torch.allclose(out_m, F.scaled_dot_product_attention(q, k, v, attn_mask=mask), atol=1e-5))

**Q3.** Why divide by $\sqrt{d}$? Run the experiment below: for random unit-variance `q`, `k`, print the standard deviation of `q . k` for `d = 16, 64, 256`, and the largest softmax weight over 64 keys with and without the scaling. What happens to gradients when the softmax is nearly one-hot?

*Your answer:*

In [ ]:
for d in (16, 64, 256):
    q, k = torch.randn(1000, 64, d), torch.randn(1000, 64, d)
    # TODO: dot products of q[:, 0] with all 64 keys, their std,
    # and the mean max softmax weight with and without / sqrt(d)
    pass

## Task 5: multi-head self-attention

One linear layer makes `q`, `k`, `v` for all heads at once; each head attends in its own `d = dim / heads` subspace; the heads are concatenated and mixed by an output projection.

- `self.qkv = nn.Linear(dim, 3 * dim)`, `self.proj = nn.Linear(dim, dim)`.
- `forward`: `(B, T, dim) -> (B, T, dim)`.
- Split `qkv(x)` as `(B, T, 3, heads, d)`, i.e. all of q first, then k, then v. That is the layout `nn.MultiheadAttention.in_proj_weight` uses, so the check can copy weights.
- Use your `attention`. Save the weights in `self.last_attn` (detached) for Task 10.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim: int, heads: int):
        super().__init__()
        assert dim % heads == 0
        self.heads, self.head_dim = heads, dim // heads
        # TODO: self.qkv, self.proj
        self.last_attn = None
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
mhsa = MultiHeadSelfAttention(192, 3)
ref = nn.MultiheadAttention(192, 3, batch_first=True)
with torch.no_grad():
    ref.in_proj_weight.copy_(mhsa.qkv.weight)
    ref.in_proj_bias.copy_(mhsa.qkv.bias)
    ref.out_proj.weight.copy_(mhsa.proj.weight)
    ref.out_proj.bias.copy_(mhsa.proj.bias)
x = torch.randn(2, 65, 192)
out = mhsa(x)
ref_out, ref_attn = ref(x, x, x, average_attn_weights=False)
check('shape', out.shape == x.shape)
check('matches nn.MultiheadAttention', torch.allclose(out, ref_out, atol=1e-5))
check('attention saved per head', mhsa.last_attn is not None and torch.allclose(mhsa.last_attn, ref_attn, atol=1e-5))

**Q4.** Count multiply-adds per block for `T` tokens and width `D` with MLP width `4D`: the four `D x D` projections (q, k, v, out), `q k^T` and `attn v`, and the MLP. For our CIFAR model (`T = 65, D = 192`), which part dominates? For ViT-B/16 at 224 (`T = 197, D = 768`)? At what `T` does attention cost as much as the MLP? Why does Swin use local windows?

*Your answer:*

## Task 6: MLP and the transformer block

- `MLP(dim, hidden)`: `fc1` (dim to hidden), `GELU`, `fc2` (hidden to dim). Use these attribute names.
- `Block(dim, heads, mlp_ratio)`, **pre-norm**:
  - `x = x + attn(norm1(x))`
  - `x = x + mlp(norm2(x))`
  - attributes `norm1`, `attn`, `norm2`, `mlp`; `hidden = int(dim * mlp_ratio)`.

In [ ]:
class MLP(nn.Module):
    def __init__(self, dim: int, hidden: int):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError


class Block(nn.Module):
    def __init__(self, dim: int, heads: int, mlp_ratio: float = 4.0):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
blk = Block(192, 3, 4.0).eval()
ref = nn.TransformerEncoderLayer(192, 3, dim_feedforward=768, dropout=0.0, activation='gelu',
                                 batch_first=True, norm_first=True).eval()
with torch.no_grad():
    ref.self_attn.in_proj_weight.copy_(blk.attn.qkv.weight)
    ref.self_attn.in_proj_bias.copy_(blk.attn.qkv.bias)
    ref.self_attn.out_proj.load_state_dict(blk.attn.proj.state_dict())
    ref.linear1.load_state_dict(blk.mlp.fc1.state_dict())
    ref.linear2.load_state_dict(blk.mlp.fc2.state_dict())
    ref.norm1.load_state_dict(blk.norm1.state_dict())
    ref.norm2.load_state_dict(blk.norm2.state_dict())
x = torch.randn(2, 65, 192)
with torch.no_grad():
    out, ref_out = blk(x), ref(x)
check('shape', out.shape == x.shape)
check('matches nn.TransformerEncoderLayer(norm_first=True)', torch.allclose(out, ref_out, atol=1e-4),
      f'max diff {(out - ref_out).abs().max():.2e}')

**Q5.** The original Transformer was post-norm: `x = norm(x + attn(x))`. Why do almost all ViTs use pre-norm? Think about what the residual path carries from the input to the output at initialisation.

*Your answer:*

## Task 7: the full ViT

Put it together. Keep these attribute names; Task 11 relies on them.

- `patch_embed` (`PatchEmbed`, with `bias=patch_bias`), `tokens` (`TokenPrep`)
- `ln_pre`: `nn.LayerNorm(dim)` if `ln_pre` else `nn.Identity()` (CLIP has one, the original ViT doesn't)
- `blocks`: `nn.ModuleList` of `depth` `Block`s
- `norm`: final `nn.LayerNorm(dim)`
- `head`: `nn.Linear(dim, num_classes, bias=head_bias)`
- `forward_features(x)`: `(B, C, H, W) -> (B, N + 1, dim)` after `norm`.
- `forward(x)`: `head` applied to the `[CLS]` output, `(B, num_classes)`.

In [ ]:
class ViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_chans=3, num_classes=10, dim=192, depth=6, heads=3,
                 mlp_ratio=4.0, patch_bias=True, ln_pre=False, head_bias=True):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward_features(self, x):
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
def expected_params(img_size=32, patch_size=4, in_chans=3, num_classes=10, dim=192, depth=6, mlp_ratio=4.0):
    D, H, N = dim, int(dim * mlp_ratio), (img_size // patch_size) ** 2
    patch = in_chans * patch_size ** 2 * D + D
    tokens = D + (N + 1) * D
    block = 2 * (2 * D) + (3 * D * D + 3 * D) + (D * D + D) + (D * H + H) + (H * D + D)
    return patch + tokens + depth * block + 2 * D + D * num_classes + num_classes


vit = ViT()
x = torch.randn(4, 3, 32, 32)
check('logits shape', vit(x).shape == (4, 10))
check('features shape', vit.forward_features(x).shape == (4, 65, 192))
n_params = sum(p.numel() for p in vit.parameters())
check('parameter count', n_params == expected_params(), f'{n_params:,} vs {expected_params():,}')
print(f'{n_params / 1e6:.2f}M parameters')

vit(x).sum().backward()
check('gradients reach cls and pos', vit.tokens.cls.grad is not None and vit.tokens.pos.grad.abs().sum() > 0)

**Sanity check every new model gets: overfit one batch.** If a model can't drive the loss to ~0 on 32 fixed random images, something is wrong (a detached tensor, a broken residual, a wrong `permute`). Run it.

In [ ]:
torch.manual_seed(0)
tiny = ViT(dim=64, depth=2, heads=2)
xb, yb = torch.randn(32, 3, 32, 32), torch.randint(0, 10, (32,))
opt = torch.optim.AdamW(tiny.parameters(), lr=3e-3)
for step in range(300):
    loss = F.cross_entropy(tiny(xb), yb)
    opt.zero_grad()
    loss.backward()
    opt.step()
print(f'final loss {loss.item():.4f}')
check('overfits one batch', loss.item() < 0.1)

**Q6. Position embeddings, experimentally.** Build `ViT(dim=64, depth=2, heads=2)`, set `tokens.pos` to zero, and compare the `[CLS]` output for an image and for the same image with its patches shuffled (shuffle `patch_embed` outputs before `tokens`, or shuffle 4x4 blocks of pixels). Then repeat with the random `pos`. What do you see, and why?

*Your answer:*

In [ ]:
# TODO: Q6 experiment


## Task 8: CIFAR-10

50k training and 10k test images, 32x32, 10 classes. ViTs have no built-in notion of locality, so on small data they lean on augmentation. Fill in `train_tf`: random crop 32 with padding 4, horizontal flip, `ToTensor`, `Normalize(mean, std)`. The test transform has no augmentation.

If the download is blocked on the devbox, use the Hugging Face copy instead: `datasets.load_dataset('uoft-cs/cifar10')` (your HF token works), and wrap it in a small `Dataset` that applies the same transforms.

In [ ]:
import torchvision
import torchvision.transforms as T

mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
train_tf = None  # TODO
test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

root = '../data/cifar10'
train_ds = torchvision.datasets.CIFAR10(root, train=True, download=True, transform=train_tf)
test_ds = torchvision.datasets.CIFAR10(root, train=False, download=True, transform=test_tf)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=8,
                                           pin_memory=True, drop_last=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=512, num_workers=8, pin_memory=True)

xb, yb = next(iter(train_loader))
check('batch shape', xb.shape == (256, 3, 32, 32))
check('normalised', abs(xb.mean().item()) < 0.3 and 0.7 < xb.std().item() < 1.3)
check('augmentation on', train_tf is not None and any(isinstance(t, T.RandomCrop) for t in train_tf.transforms))

## Task 9: the training loop

Four pieces, the same ones `scripts/train_theta.py` uses.

1. **`param_groups(model, weight_decay)`**: two AdamW groups. Weight decay on weight matrices and conv kernels; **none** on biases, LayerNorm parameters, `[CLS]` and position embeddings. Note `pos` is 3-D, so a plain `p.ndim < 2` rule gets it wrong.
2. **`cosine_with_warmup(step, total, warmup)`**: an LR multiplier rising linearly to 1 over `warmup` steps, then a cosine down to 0 at `total`.
3. **`train_one_epoch(model, loader, opt, sched, device, label_smoothing=0.1)`**: `bf16` autocast on GPU, cross-entropy with label smoothing, gradient clipping at 1.0, `sched.step()` every batch. Return `(mean loss, accuracy)`.
4. **`evaluate(model, loader, device)`**: `model.eval()`, no gradients, return accuracy.

In [ ]:
def param_groups(model, weight_decay):
    # TODO
    raise NotImplementedError


def cosine_with_warmup(step, total, warmup):
    # TODO
    raise NotImplementedError


def train_one_epoch(model, loader, opt, sched, device, label_smoothing=0.1):
    # TODO
    raise NotImplementedError


@torch.no_grad()
def evaluate(model, loader, device):
    # TODO
    raise NotImplementedError

In [ ]:
m = ViT(dim=64, depth=2, heads=2)
decay, no_decay = param_groups(m, 0.05)
ids = lambda ps: {id(p) for p in ps}
check('two groups', decay['weight_decay'] == 0.05 and no_decay['weight_decay'] == 0.0)
check('pos and cls not decayed', {id(m.tokens.pos), id(m.tokens.cls)} <= ids(no_decay['params']))
check('norms and biases not decayed', {id(m.norm.weight), id(m.blocks[0].mlp.fc1.bias)} <= ids(no_decay['params']))
check('weights decayed', {id(m.patch_embed.proj.weight), id(m.blocks[0].attn.qkv.weight)} <= ids(decay['params']))
check('nothing missing', len(decay['params']) + len(no_decay['params']) == len(list(m.parameters())))

check('warmup', abs(cosine_with_warmup(0, 100, 10) - 0.1) < 1e-9 and abs(cosine_with_warmup(9, 100, 10) - 1.0) < 1e-9)
check('cosine', abs(cosine_with_warmup(55, 100, 10) - 0.5) < 1e-9 and cosine_with_warmup(100, 100, 10) < 1e-9)

**Train.** On one V100 an epoch takes roughly 15 to 30 seconds. Expect test accuracy in the 70s after 30 epochs; a ResNet-18 gets ~93% on CIFAR-10, and Q7 asks why the gap exists. Try changing one thing at a time afterwards: `patch_size=2` (4x the tokens), no augmentation, no warmup, `depth=12`.

In [ ]:
EPOCHS, LR, WD, WARMUP_EPOCHS = 30, 1e-3, 0.05, 3
cfg = dict(img_size=32, patch_size=4, num_classes=10, dim=192, depth=6, heads=3, mlp_ratio=2.0)

torch.manual_seed(0)
model = ViT(**cfg).to(device)
opt = torch.optim.AdamW(param_groups(model, WD), lr=LR)
total = EPOCHS * len(train_loader)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: cosine_with_warmup(s, total, WARMUP_EPOCHS * len(train_loader)))

for epoch in range(EPOCHS):
    t0 = time.time()
    loss, acc = train_one_epoch(model, train_loader, opt, sched, device)
    test_acc = evaluate(model, test_loader, device)
    print(f'epoch {epoch + 1:>2}  loss {loss:.3f}  train {acc:.3f}  test {test_acc:.3f}  {time.time() - t0:.0f}s')
torch.save(model.state_dict(), 'vit_cifar.pt')

**Q7.** Why does a ViT trained from scratch on 50k images lose to a small CNN, while pretrained ViTs beat CNNs? What does a convolution assume about images that attention has to learn from data?

*Your answer:*

## Task 10: look inside

1. **`cls_attention_map(model, img)`**: run one image `(3, 32, 32)` through `model`, take the last block's `attn.last_attn`, keep the `[CLS]` query row over the patch keys, average over heads, and reshape to the `(8, 8)` patch grid.
2. **`pos_similarity(model)`**: cosine similarity between the patch position embeddings (`pos[0, 1:]`), shape `(64, 64)`.

The plotting is given. If the position embeddings learned 2-D structure, each patch's map lights up its neighbours, although nothing told the model the patches form a grid.

In [ ]:
@torch.no_grad()
def cls_attention_map(model, img):
    # TODO
    raise NotImplementedError


@torch.no_grad()
def pos_similarity(model):
    # TODO
    raise NotImplementedError

In [ ]:
import matplotlib.pyplot as plt

model.eval()
classes = test_ds.classes
unnorm = lambda t: (t * torch.tensor(std).view(3, 1, 1) + torch.tensor(mean).view(3, 1, 1)).clamp(0, 1)
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for i in range(6):
    img, label = test_ds[i]
    amap = cls_attention_map(model, img.to(device)).float().cpu()
    check(f'map {i} shape', amap.shape == (8, 8))
    axes[0, i].imshow(unnorm(img).permute(1, 2, 0))
    axes[0, i].set_title(classes[label])
    axes[1, i].imshow(amap, cmap='viridis')
    for ax in axes[:, i]:
        ax.axis('off')
plt.suptitle('[CLS] attention in the last block')
plt.show()

sim = pos_similarity(model).float().cpu()
check('similarity shape', sim.shape == (64, 64))
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, idx in zip(axes, (0, 7, 27, 36, 63)):
    ax.imshow(sim[idx].reshape(8, 8), cmap='viridis')
    ax.set_title(f'patch {idx} (row {idx // 8}, col {idx % 8})')
    ax.axis('off')
plt.suptitle('cosine similarity of position embeddings')
plt.show()

## Task 11: load real CLIP ViT-B/32 weights into your ViT

If your blocks are right, they can run CLIP's weights. Write `load_open_clip(vit, visual)` that copies `open_clip`'s ViT-B/32 into `ViT(img_size=224, patch_size=32, num_classes=512, dim=768, depth=12, heads=12, patch_bias=False, ln_pre=True, head_bias=False)`.

Where things live in `clip.visual` (print it to see):

- `conv1` (no bias), `class_embedding` `(768,)`, `positional_embedding` `(50, 768)`, `ln_pre`
- `transformer.resblocks[i]`: `ln_1`, `attn` (an `nn.MultiheadAttention`), `ln_2`, `mlp.c_fc`, `mlp.c_proj`
- `ln_post`, and `proj`: a `(768, 512)` matrix applied as `x @ proj`. Your `head` is a `Linear`, which computes `x @ W.T`.

When it matches, open `scripts/vit_encoder.py`. It does the same walk through `clip.visual`, but reuses the layers instead of copying them, so it can freeze the first blocks and fine-tune the last few.

In [ ]:
@torch.no_grad()
def load_open_clip(vit, visual):
    # TODO
    raise NotImplementedError

In [ ]:
import open_clip

clip, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
clip.eval()
mine = ViT(img_size=224, patch_size=32, num_classes=512, dim=768, depth=12, heads=12,
           patch_bias=False, ln_pre=True, head_bias=False).eval()
load_open_clip(mine, clip.visual)

x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    ref, out = clip.encode_image(x), mine(x)
print(f'max abs diff {(out - ref).abs().max():.2e}')
check('matches open_clip encode_image', torch.allclose(out, ref, atol=1e-4))
print(f'{sum(p.numel() for p in mine.parameters()) / 1e6:.1f}M parameters')

## Task 12 (stretch): pretrained vs from scratch

Resize CIFAR-10 to 224 (`T.Resize(224)` before `ToTensor`, CLIP's mean and std) and compare, on the same test set:

1. **Linear probe**: frozen CLIP features (`mine.forward_features(x)[:, 0]`), train only a new `Linear(768, 10)`.
2. **Fine-tune the last 2 blocks** plus the head; everything else frozen. Use a small LR (1e-5) for the blocks and a larger one (1e-3) for the head: two param groups.
3. **Full fine-tune** with the same two learning rates.

Record accuracy and time per epoch for each, next to your from-scratch number from Task 9.

**Q8.** This is the same choice we face for Stage 1: frozen CLIP features with a small head, or fine-tune the last blocks on watch-fraction distributions. Given ~10k labelled videos (not 50k images), which would you try first, and what would convince you to fine-tune?

*Your answer:*

In [ ]:
# TODO: Task 12
